# Notebook 04 — Final fits and calibration

**Purpose.** Refit the candidate and the locked comparator on train+tune for the frozen epoch count with seeds 101/202/303 (sequential, one GPU job, at most 70 min each), then fit calibration objects. **Partitions:** train+tune for weights; calibration-T for temperature; calibration-P for referral thresholds (both handled by the evaluator pipeline in notebook 05, never used for gradient fitting). **GPU:** final allocation.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## GPU lock, budget check and frozen configuration

In [ ]:
from cape_eeg.training.supervisor import GpuLedger
lock = read_json(ws.manifests / 'protocol_lock.json'); led = GpuLedger(ws.runs / 'gpu_ledger.jsonl')
print('ledger used %.2f h of 12; remaining %.2f h' % (led.total_hours(), led.remaining_hours())); print({k: lock[k] for k in ['candidate','comparator','final_epochs','final_seeds','deployment_seed','candidate_lr_mult','comparator_lr_mult']})

## Six sequential final fits (plus the exploratory P+MSF if budget allows)

In [ ]:
import subprocess
needed = [(cfg, s) for cfg in [lock['candidate'], lock['comparator'], 'P_MSF'] for s in lock['final_seeds']]
done = {(read_json(rd / 'config.resolved.json')['config_id'], read_json(rd / 'config.resolved.json')['seed']) for rd in ws.runs.glob('final_*') if (read_json(rd / 'status.json') or {}).get('status') == 'PASS'}
missing = [x for x in needed if x not in done]
if missing:
    env = dict(os.environ, EXPLORATORY='P_MSF', SKIP_EVALUATION='1')
    p = subprocess.run(['bash', str(REPO / 'scripts' / 'run_final_pipeline.sh')], capture_output=True, text=True, env=env)
    print('\n'.join(l for l in (p.stdout + p.stderr).splitlines() if l.strip() and 'Warning' not in l)); assert p.returncode == 0, 'final pipeline failed'
else:
    print('all', len(done), 'final fits already PASS with matching hashes; reused (no silent retraining)')

## Completion manifests

In [ ]:
for rd in sorted(ws.runs.glob('final_*')):
    st = read_json(rd / 'status.json'); cfg = read_json(rd / 'config.resolved.json')
    print(cfg['config_id'], 'seed', cfg['seed'], st['status'], 'complete_schedule', st.get('complete_schedule'), 'epochs', st.get('epochs_completed'), 'wall %.1f min' % (st.get('wall_seconds', 0) / 60), 'peak CUDA %.2f GiB' % st.get('peak', {}).get('cuda_alloc_gib', 0))